In [2]:
import sys
!{sys.executable} -m pip install langchain-mcp-adapters


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
print(sys.executable)

c:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe


In [10]:
import sys 
import os 
import io 
import asyncio
from dotenv import load_dotenv 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import sys
import os
import io
import asyncio
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

# Fix Windows asyncio subprocess loop policy
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Fix Jupyter Notebook (ipykernel) sys.stderr fileno issue on Windows
try:
    import ipykernel.iostream
    ipykernel.iostream.OutStream.fileno = lambda self: sys.__stderr__.fileno() if hasattr(sys.__stderr__, 'fileno') else 2
except Exception:
    pass
try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    sys.stderr.fileno = lambda: devnull_fd 

In [11]:
load_dotenv()

True

In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [13]:
client = MultiServerMCPClient(
    {
        "student_database":{
            "transport": "stdio",
            "command": "python",
            "args":["mcp_server.py"]
        }
    }
)

In [15]:
tools = await client.get_tools()

  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py", line 3697, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\hp\AppData\Local\Temp\ipykernel_16964\229651158.py", line 1, in <module>
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_mcp_adapters\client.py", line 213, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_mcp_adapters\tools.py", line 590, in load_mcp_tools
  |     async with create_session(
  |                ~~~~~~~~~~~~~~^
  |         connection, mcp_callbacks=mcp_callbacks
  |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful student assistant.

    Use the student database tool whenever
    the user asks about student information.

    Do not make up student information. 
    Use the database data to answer.
    """
)

In [ ]:
question = input("Ask your question")

In [ ]:
result = await agent.invoke({
    "messages":[
        ("user",question)
    ]
})